**AERISBQ - Algorithmic Entropy Reduction in Sequenced Binary Quadratics.**

This notebook asks a single question about a target number $N$:

> Are there two 2-bit numbers $a$ and $b$, each drawn from
> $\{0, 1, 2, 3\}$, whose product is exactly $N$?

**The classical approach.**
There are only $4 \times 4 = 16$ candidate pairs, so a classical computer
simply tries all of them.
That is trivial at this size, but the cost of brute force grows as
$2^{2n}$ for $n$-bit factors.
At $n = 1024$ there is no machine that can enumerate the candidates, and
that is precisely why factoring large numbers is hard.

**The quantum approach used here.**

1.) Put $a$ and $b$ into equal superposition with four Hadamard gates.
    The register now holds all 16 $(a, b)$ pairs simultaneously.

2.) Run a **reversible binary multiplier** built entirely from CNOT and
    Toffoli gates. Every branch of the superposition computes its own
    4-bit product $p = a \times b$.

3.) Run an **oracle** that compares $p$ against $N$ and raises a single
    flag qubit on the branches where they match.

4.) Measure the flag qubit.

**Reading the name against the circuit.**

- *Binary quadratics* - the quantity being solved for is $a \times b$,
  which is quadratic in the unknown bits, and both operands are binary.
  Fixing the product and solving for the operands is the quadratic
  equation this circuit attacks.
- *Sequenced* - the multiplier does not evaluate that quadratic in one
  step. It runs as a fixed sequence of stages: partial products first,
  then the weight-2 column, then the weight-4 column, each stage feeding
  the next through ancilla qubits.
- *Entropy reduction* - the four Hadamards deliberately start the register
  at maximum entropy, 4 bits of it spread evenly over all 16 candidate
  pairs. Everything after that narrows it. The multiplier maps each
  candidate to its product, the oracle collapses that comparison onto a
  single qubit, and measuring it converts a 16-way uncertainty into one
  bit of information about $N$.
- *Algorithmic* - the reduction is done by construction rather than by
  search. No branch is ever examined individually.

**Why the probabilities come out exact.**
After the opening layer of Hadamards the circuit contains nothing but
CNOT, Toffoli, and X gates: pure reversible classical logic with no phase
gates anywhere.
The 16 branches keep distinct values on the input and ancilla qubits, so
they stay orthogonal and never recombine.
No amplitudes cancel, and there is no interference of any kind.
The flag therefore reads 1 with the exact probability

$$P(\text{flag}=1) = \frac{\text{number of pairs } (a, b) \text{ with } ab = N}{16}$$

**How this relates to `satisfiability.ipynb`.**
That notebook evaluated a small Boolean formula across all assignments at
once.
This one does the same thing with a larger formula: "$a \times b = N$".
Factoring is a satisfiability question wearing an arithmetic costume, and
the circuit below makes that literal.

**The input qubits are never copied.**
$a_0$, $a_1$, $b_0$, and $b_1$ appear only as **controls** of Toffoli
gates.
A control passes through its gate unchanged; only the ancilla target
flips.
Using a qubit as a control is categorically different from fanning it out,
so nothing here violates the no-cloning theorem.


**This notebook targets $N = 9$.**
$9 = 3 \times 3$ is the largest product two 2-bit numbers can make, and it
is reached in exactly one way.
Because $a = b$ here, the two orderings coincide rather than giving two
branches as they did for $N = 6$.
This is the **satisfiable** case with a single solution:
$P(\text{flag}=1) = 1/16 = 6.25\%$.

9 is also the only target of the four that exercises the final carry.
It is the one product whose $p_3$ bit is set, so it is the only case where
the top Toffoli of the ripple adder does anything at all.

---
**Cell 01 - Imports and shared backend.**
A single `AerSimulator` backend is shared by every cell below.
`TARGET` is the number this notebook tries to factor, and it is the only
value that differs between the four notebooks in this folder.

In [ ]:
"""aerisbq_9.ipynb"""

# Cell 01 - Imports and shared backend

import numpy as np
import qiskit
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_distribution
from qiskit_aer import AerSimulator

# The target product this notebook searches for. This is the only value
# that differs between the four notebooks in this folder
TARGET = 9

# One AerSimulator backend is shared by every cell below
backend = AerSimulator()

print(f"Qiskit version : {qiskit.__version__}")
print(f"Backend        : {backend.name}")
print(f"Target product : {TARGET} (binary {TARGET:04b})")

---
**Cell 02 - The 15-qubit register.**
Bare qubit indices make a circuit this size unreadable, so every qubit
gets a name.
The register splits into four groups:

| Qubits | Role |
| --- | --- |
| $q_0, q_1$ | the input $a = 2a_1 + a_0$ |
| $q_2, q_3$ | the input $b = 2b_1 + b_0$ |
| $q_4 \ldots q_7$ | the four partial products $a_i b_j$ |
| $q_8 \ldots q_{11}$ | the ripple-adder sums and carries |
| $q_{12}, q_{13}$ | oracle comparison ancillas |
| $q_{14}$ | the flag, 1 exactly when $a \times b = N$ |

**Where the product bits come from.**
Writing $a = 2a_1 + a_0$ and $b = 2b_1 + b_0$ and expanding the product:

$$a \times b = 4\,(a_1b_1) + 2\,(a_1b_0 + a_0b_1) + a_0b_0$$

Reading that column by column gives the four product bits:

- $p_0 = a_0b_0$, the weight-1 column, which has no carry into it
- $p_1 = a_0b_1 \oplus a_1b_0$, the sum bit of the weight-2 column
- $p_2 = c_1 \oplus a_1b_1$, where $c_1 = a_0b_1 \wedge a_1b_0$ is the
  carry out of the weight-2 column
- $p_3 = c_2 = c_1 \wedge a_1b_1$, the final carry

$p_0$ and $p_3$ are not separate qubits.
They are aliases for $q_4$ and $q_{11}$, which already hold those values,
which is why 15 qubits are enough.

In [ ]:
# Cell 02 - Name every qubit in the 15-qubit register

# Inputs: a = 2*a1 + a0 and b = 2*b1 + b0, each in the range 0..3
Q_A0, Q_A1 = 0, 1
Q_B0, Q_B1 = 2, 3

# The four partial products, one per pair of input bits
Q_A0B0, Q_A0B1, Q_A1B0, Q_A1B1 = 4, 5, 6, 7

# Ripple-adder working qubits
Q_S1 = 8  # sum   of a0b1 + a1b0
Q_C1 = 9  # carry of a0b1 + a1b0
Q_S2 = 10  # sum   of c1 + a1b1
Q_C2 = 11  # carry of c1 + a1b1

# The 4-bit product p = 8*p3 + 4*p2 + 2*p1 + p0. Note that p0 and p3 are
# aliases: those values already live on Q_A0B0 and Q_C2
Q_P0, Q_P1, Q_P2, Q_P3 = Q_A0B0, Q_S1, Q_S2, Q_C2

# Oracle working qubits and the final answer qubit
Q_M01 = 12  # 1 when the low  half of the product matches TARGET
Q_M23 = 13  # 1 when the high half of the product matches TARGET
Q_FLAG = 14  # 1 exactly when the full product equals TARGET

N_QUBITS = 15

print(f"Register width : {N_QUBITS} qubits")
print(f"Inputs         : a = (q{Q_A1}, q{Q_A0}),  b = (q{Q_B1}, q{Q_B0})")
print(f"Product bits   : p3=q{Q_P3}  p2=q{Q_P2}  p1=q{Q_P1}  p0=q{Q_P0}")
print(f"Flag qubit     : q{Q_FLAG}")

---
**Cell 03 - The reversible 2x2 multiplier.**
Two gates do all the work:

- **Toffoli (CCX)** flips its target when both controls are 1, so
  `ccx(x, y, t)` with $t$ starting at $\lvert 0\rangle$ leaves
  $t = x \wedge y$. That is a **logical AND**.
- **CNOT (CX)** flips its target when its control is 1, so two CNOTs into
  the same fresh target leave $t = x \oplus y$. That is a **logical XOR**.

An XOR for the sum bit plus an AND for the carry bit is exactly a half
adder, and the multiplier is two half adders stacked on top of the four
partial products.

Both gates are reversible and neither introduces a relative phase, so this
entire block is ordinary classical logic executed on a quantum register.
The only quantum thing about it is that it runs on all 16 inputs at once.

The cell ends by checking the circuit against ordinary arithmetic for all
16 input pairs, which is worth doing before trusting anything the oracle
says about it.

In [ ]:
# Cell 03 - The reversible 2x2 binary multiplier


def build_multiplier() -> QuantumCircuit:
    """Return the 15-qubit circuit that multiplies a by b.

    The four input qubits pass through untouched; the 4-bit product lands
    on Q_P0, Q_P1, Q_P2, Q_P3. Every gate is a CNOT or a Toffoli, so the
    whole block is reversible classical logic with no interference
    """
    qc = QuantumCircuit(N_QUBITS)

    # The four partial products a_i * b_j, each an AND of two input bits
    qc.ccx(Q_A0, Q_B0, Q_A0B0)
    qc.ccx(Q_A0, Q_B1, Q_A0B1)
    qc.ccx(Q_A1, Q_B0, Q_A1B0)
    qc.ccx(Q_A1, Q_B1, Q_A1B1)

    # Weight-2 column: a0b1 + a1b0 -> sum on Q_S1 (XOR), carry on Q_C1 (AND)
    qc.cx(Q_A0B1, Q_S1)
    qc.cx(Q_A1B0, Q_S1)
    qc.ccx(Q_A0B1, Q_A1B0, Q_C1)

    # Weight-4 column: c1 + a1b1 -> sum on Q_S2 (XOR), carry on Q_C2 (AND)
    qc.cx(Q_C1, Q_S2)
    qc.cx(Q_A1B1, Q_S2)
    qc.ccx(Q_C1, Q_A1B1, Q_C2)

    return qc


def load_inputs(qc: QuantumCircuit, a: int, b: int) -> None:
    """Set the four input qubits to the classical bit patterns of a and b"""
    bits = [a & 1, (a >> 1) & 1, b & 1, (b >> 1) & 1]
    for bit, qubit in zip(bits, [Q_A0, Q_A1, Q_B0, Q_B1]):
        if bit:
            qc.x(qubit)


def read_product(index: int) -> int:
    """Decode the 4-bit product from a computational basis state index.

    Qiskit is little-endian, so qubit i is bit i of the basis state index
    """
    p0 = (index >> Q_P0) & 1
    p1 = (index >> Q_P1) & 1
    p2 = (index >> Q_P2) & 1
    p3 = (index >> Q_P3) & 1
    return 8 * p3 + 4 * p2 + 2 * p1 + p0


# Check the multiplier against ordinary arithmetic for all 16 input pairs.
# Each run starts from a single basis state and uses only classical logic,
# so exactly one amplitude survives and argmax finds it
print("  a   b   circuit   a*b")
all_correct = True
for a in range(4):
    for b in range(4):
        test = QuantumCircuit(N_QUBITS)
        load_inputs(test, a, b)
        test.compose(build_multiplier(), inplace=True)

        basis_index = int(np.argmax(np.abs(Statevector(test).data)))
        product = read_product(basis_index)

        all_correct = all_correct and product == a * b
        print(f"  {a}   {b}   {product:7d}   {a * b:3d}")

print()
print(f"All 16 input pairs multiply correctly: {all_correct}")

---
**Cell 04 - The oracle and the assembled circuit.**
The oracle has to answer "does $p$ equal $N$?" using gates that only know
how to fire when their controls read **1**.

The trick is to flip the product bits that $N$ requires to be 0.
Here $N = 9$ is $1001_2$, so we need
$p_3 = 1$, $p_2 = 0$, $p_1 = 0$, $p_0 = 1$, and an X gate goes
on $p_1$ and $p_2$.
After those flips the statement "all four product qubits read 1" and the
statement "the product equals $N$" are the same statement, so a small tree
of three Toffoli gates collapses the four-way AND onto the flag qubit.

The X flips are then **uncomputed** by a second identical layer.
An X gate is its own inverse, so this costs two gates and leaves the
product qubits holding the true product, which Cell 06 reads back to
recover the actual factor pairs.
Uncomputing scratch work like this is standard practice in reversible
circuit design: an ancilla left in a modified state stays entangled with
the rest of the register and cannot safely be reused.

The assembled circuit is four Hadamards, then the multiplier, then the
oracle, then a single measurement of the flag qubit onto the one classical
bit.

In [ ]:
# Cell 04 - The oracle, and the full circuit assembled

# TARGET in binary, least significant bit first: [p0, p1, p2, p3]
TARGET_BITS = [(TARGET >> k) & 1 for k in range(4)]
PRODUCT_QUBITS = [Q_P0, Q_P1, Q_P2, Q_P3]


def add_oracle(qc: QuantumCircuit) -> None:
    """Set Q_FLAG to 1 exactly on the branches whose product equals TARGET.

    A Toffoli fires only when both of its controls read 1, so any product
    bit that TARGET requires to be 0 is flipped first. After the flips,
    "all four product qubits read 1" means "the product matches TARGET"
    """
    # Flip the product bits that TARGET requires to be 0
    for qubit, bit in zip(PRODUCT_QUBITS, TARGET_BITS):
        if bit == 0:
            qc.x(qubit)

    # Collapse the four-way AND with a small tree of Toffoli gates
    qc.ccx(Q_P0, Q_P1, Q_M01)  # low  half of the product matches
    qc.ccx(Q_P2, Q_P3, Q_M23)  # high half of the product matches
    qc.ccx(Q_M01, Q_M23, Q_FLAG)  # therefore all four bits match

    # Uncompute the flips so the product qubits hold the true product again.
    # X is its own inverse, so an identical layer undoes the first one
    for qubit, bit in zip(PRODUCT_QUBITS, TARGET_BITS):
        if bit == 0:
            qc.x(qubit)


qc = QuantumCircuit(N_QUBITS, 1)

# Equal superposition over a and b: all 16 (a, b) pairs at once
qc.h(Q_A0)
qc.h(Q_A1)
qc.h(Q_B0)
qc.h(Q_B1)

qc.compose(build_multiplier(), inplace=True)
add_oracle(qc)

qc.measure(Q_FLAG, 0)  # flag qubit -> the single classical bit

display(qc.draw(output="mpl"))
flips = [f"p{k}" for k in range(4) if TARGET_BITS[k] == 0]
print(f"Target {TARGET} = binary {TARGET:04b}")
print(
    f"Required product bits : p3={TARGET_BITS[3]} p2={TARGET_BITS[2]} "
    f"p1={TARGET_BITS[1]} p0={TARGET_BITS[0]}"
)
print(f"X gates applied to    : {', '.join(flips) if flips else 'none'}")

---
**Cell 05 - Sampling the flag qubit.**
1024 shots, matching the other notebooks in this session.


One of the 16 branches is flagged, so roughly 64 of the 1024 shots should
return 1: the rarest of the four targets in this folder.

That rarity is the point in miniature.
Halving the number of solutions halved the number of hits, and in a real
factoring problem the two prime factors are a vanishing fraction of an
astronomically large search space.
The circuit still evaluates every candidate in one execution; it is the
*readout* that becomes hopeless, which is exactly the gap Grover's
algorithm was invented to narrow.

In [ ]:
# Cell 05 - Sample the flag qubit

SHOTS = 1024

result = backend.run(transpile(qc, backend), shots=SHOTS).result()
counts = result.get_counts(qc)

display(plot_distribution(counts))

hits = counts.get("1", 0)
print(f"Shots           : {SHOTS}")
print(f"Flag read as 1  : {hits}  ({100 * hits / SHOTS:.2f}%)")
print()
if hits:
    print(f"At least one pair of 2-bit numbers multiplies to {TARGET}")
    print("The target is SATISFIABLE, settled by a single observed 1")
else:
    print(f"No shot found a pair of 2-bit numbers multiplying to {TARGET}")
    print("That is consistent with UNSATISFIABLE but does not prove it,")
    print("because a rare solution would also be missed. See Cell 06")

---
**Cell 06 - The exact answer, with no sampling at all.**
Everything before the measurement is a unitary acting on the fixed input
$\lvert 0 \rangle^{\otimes 15}$, so the state it prepares is one definite
vector of $2^{15} = 32768$ amplitudes.
Simulating it hands us those amplitudes algebraically, with no statistical
error whatsoever.

**Where to look for a flag.**
Qiskit is little-endian, so qubit $i$ is bit $i$ of the basis-state index.
The flag qubit $q_{14}$ is therefore bit 14, and the question "can the
flag ever read 1?" becomes a search:

> Does any basis state whose bit 14 is set carry a nonzero amplitude?

If even one does, the target is a product of two 2-bit numbers.
If every one of them is exactly zero, reading a 1 is structurally
impossible and the target is not, proven with certainty rather than
estimated from shots.

Every flagged index also carries the $a$ and $b$ that produced it, on bits
0 through 3, so the same scan that answers the yes/no question also prints
the factor pairs themselves.

**This is not a speedup.**
The scan costs $O(2^n)$, which is exactly the brute-force enumeration the
quantum device avoids at run time.
It is a teaching instrument: it lets us see the exact answer that real
hardware could only ever sample.

In [ ]:
# Cell 06 - Read the exact answer off the statevector, with no shots

# Operator/Statevector need a purely unitary circuit, so drop the measurement
qc_unitary = qc.remove_final_measurements(inplace=False)
state = Statevector(qc_unitary).data

TOL = 1e-9

solutions: list[tuple[int, int, int]] = []
flag_probability = 0.0
for index, amplitude in enumerate(state):
    if (index >> Q_FLAG) & 1 == 0:
        continue
    flag_probability += abs(amplitude) ** 2
    if abs(amplitude) <= TOL:
        continue
    # A flagged basis state still carries the a and b that produced it
    a = 2 * ((index >> Q_A1) & 1) + ((index >> Q_A0) & 1)
    b = 2 * ((index >> Q_B1) & 1) + ((index >> Q_B0) & 1)
    solutions.append((a, b, read_product(index)))

print(f"Basis states in the statevector : {len(state)}")
print(f"...with the flag qubit set      : {2 ** (N_QUBITS - 1)}")
print(f"...of those, carrying amplitude : {len(solutions)}")
print(f"P(flag = 1)                     : {flag_probability:.6f}")
print()
if solutions:
    print(f"{TARGET} IS the product of two 2-bit numbers:")
    for a, b, product in sorted(solutions):
        print(f"  a = {a}, b = {b}  ->  a * b = {product}")
else:
    print("No flagged basis state carries any amplitude.")
    print(f"P(flag = 1) = 0 exactly, so no pair (a, b) in 0..3 gives {TARGET}")
    print("The target is UNSATISFIABLE, proven rather than merely unobserved")

---
**Result for $N = 9$.**
Exactly one basis state carries the flag, giving
$P(\text{flag}=1) = 1/16$ exactly, and the scan recovers $3 \times 3$.

Notice that this is the only target of the four whose product needs all
four bits.
$9 = 1001_2$ sets $p_3$, so the final carry Toffoli, idle for every other
target, is what makes this case work.

**What this circuit does and does not buy you.**

*What it does.*
A single execution evaluates $a \times b$ across all 16 input pairs and
marks the matches.
The gate count grows polynomially with the number of bits in the factors,
while the number of pairs being evaluated grows as $2^{2n}$.
That part is genuinely free.

*What it does not.*
Reading the answer back out is the entire problem.
The flag reads 1 with probability $k / 2^{2n}$, where $k$ is the number of
factor pairs.
For a real number with two large prime factors $k$ is 2 or 4 against an
astronomically large $2^{2n}$, so you would sample essentially forever
without ever seeing a 1.
Superposition gets the answer **into** the state; it does not get it
**out**.

**The two ways forward.**

- **Grover's algorithm** amplifies the flagged branches, turning roughly
  $2^{2n}$ shots into roughly $\sqrt{2^{2n}} = 2^n$.
  That is a quadratic gain, and still exponential in $n$.
  See `grover.ipynb`.
- **Shor's algorithm** abandons this circuit shape entirely.
  Rather than testing candidate factors it uses the quantum Fourier
  transform to find the *period* of $x \mapsto a^x \bmod N$, from which a
  factor falls out by classical arithmetic.
  That lets genuine interference do the work, and it is polynomial in $n$.
  See `qpe.ipynb` for the phase-estimation machinery Shor is built on.

The circuit in this notebook is not a competitive factoring algorithm.
It is the clearest available picture of the gap between *computing* an
answer in superposition and *learning* what that answer is.